# 01 Data Extraction

**Project:** Predictive Insights into Banking Customer Churn  
**Objective:** Load the original banking churn dataset, validate its structure, and document the raw data profile before cleaning.

This notebook is the starting point of the ETL workflow. No cleaning or transformation is performed here; the purpose is to inspect the source data and confirm that it is ready for the cleaning stage.


## 1. Import Libraries

Import the required libraries for reading and inspecting the dataset.


In [1]:
# Import required libraries for data extraction and initial validation.
import pandas as pd
import numpy as np



## 2. Load Raw Dataset

Load the original churn dataset from the project `data` folder.


In [2]:
# Define the source file path for the raw dataset.
source_path = '../data/churn_prediction.csv'

# Load the raw banking customer churn dataset.
df = pd.read_csv(source_path)

# Display the first five records to confirm successful loading.
df.head()



,customer_id,vintage,age,gender,dependents,occupation,city,customer_nw_category,branch_code,current_balance,...,average_monthly_balance_prevQ,average_monthly_balance_prevQ2,current_month_credit,previous_month_credit,current_month_debit,previous_month_debit,current_month_balance,previous_month_balance,churn,last_transaction
0,1,2101,66,Male,0.0,self_employed,187.0,2,755,1458.71,...,1458.71,1449.07,0.20,0.20,0.20,0.20,1458.71,1458.71,0,2019-05-21
1,2,2348,35,Male,0.0,self_employed,NaN,2,3214,5390.37,...,7799.26,12419.41,0.56,0.56,5486.27,100.56,6496.78,8787.61,0,2019-11-01
2,4,2194,31,Male,0.0,salaried,146.0,2,41,3913.16,...,4910.17,2815.94,0.61,0.61,6046.73,259.23,5006.28,5070.14,0,NaT
3,5,2329,90,NaN,NaN,self_employed,1020.0,2,582,2291.91,...,2084.54,1006.54,0.47,0.47,0.47,2143.33,2291.91,1669.79,1,2019-08-06
4,6,1579,42,Male,2.0,self_employed,1494.0,3,388,927.72,...,1643.31,1871.12,0.33,714.61,588.62,1538.06,1157.15,1677.16,1,2019-11-03


## 3. Dataset Overview

Check the number of rows, columns, column names, and basic structure of the raw dataset.


In [3]:
# Print the dataset shape and column list for quick structural validation.
print(f'Dataset Source: {source_path}')
print(f'Total Rows: {df.shape[0]}')
print(f'Total Columns: {df.shape[1]}')

print('\nColumn Names:')
for col in df.columns:
    print('-', col)



Dataset Source: ../data/churn_prediction.csv
Total Rows: 28382
Total Columns: 21

Column Names:
- customer_id
- vintage
- age
- gender
- dependents
- occupation
- city
- customer_nw_category
- branch_code
- current_balance
- previous_month_end_balance
- average_monthly_balance_prevQ
- average_monthly_balance_prevQ2
- current_month_credit
- previous_month_credit
- current_month_debit
- previous_month_debit
- current_month_balance
- previous_month_balance
- churn
- last_transaction


## 4. Data Types and Schema

Review each column's data type so the cleaning notebook can handle numeric, categorical, and date columns correctly.


In [4]:
# Create a schema table showing column names, data types, and non-null counts.
schema_summary = pd.DataFrame({
    'column_name': df.columns,
    'data_type': df.dtypes.astype(str).values,
    'non_null_count': df.notnull().sum().values,
    'null_count': df.isnull().sum().values
})

# Display the schema summary table.
schema_summary



,column_name,data_type,non_null_count,null_count
0,customer_id,int64,28382,0
1,vintage,int64,28382,0
2,age,int64,28382,0
3,gender,object,27857,525
4,dependents,float64,25919,2463
5,occupation,object,28302,80
6,city,float64,27579,803
7,customer_nw_category,int64,28382,0
8,branch_code,int64,28382,0
9,current_balance,float64,28382,0


## 5. Missing Value Audit

Identify missing values in the raw dataset before any imputation or transformation.


In [5]:
# Calculate missing value count and percentage for each column.
missing_summary = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_percentage': (df.isnull().mean() * 100).round(2)
}).sort_values(by='missing_count', ascending=False)

# Display only columns with missing values for a focused audit.
missing_summary[missing_summary['missing_count'] > 0]



,missing_count,missing_percentage
dependents,2463,8.68
city,803,2.83
gender,525,1.85
occupation,80,0.28


## 6. Duplicate and Target Checks

Check duplicate records and validate the target variable used for churn analysis.


In [6]:
# Count duplicate rows in the raw dataset.
duplicate_rows = df.duplicated().sum()

# Calculate churn distribution and churn rate.
churn_counts = df['churn'].value_counts().rename(index={0: 'Retained', 1: 'Churned'})
churn_rate = df['churn'].mean() * 100

# Print duplicate and target checks.
print(f'Duplicate Rows: {duplicate_rows}')
print('\nChurn Distribution:')
print(churn_counts)
print(f'\nOverall Churn Rate: {churn_rate:.2f}%')



Duplicate Rows: 0

Churn Distribution:
churn
Retained    23122
Churned      5260
Name: count, dtype: int64

Overall Churn Rate: 18.53%


## 7. Numeric Column Summary

Review descriptive statistics for numeric fields to understand scale, spread, and possible outliers before cleaning.


In [7]:
# Generate descriptive statistics for all numeric columns.
numeric_summary = df.describe().T

# Display numeric summary rounded for readability.
numeric_summary.round(2)



,count,mean,std,min,25%,50%,75%,max
customer_id,28382.0,15143.51,8746.45,1.00,7557.25,15150.50,22706.75,30301.00
vintage,28382.0,2091.14,272.68,73.00,1958.00,2154.00,2292.00,2476.00
age,28382.0,48.21,17.81,1.00,36.00,46.00,60.00,90.00
dependents,25919.0,0.35,1.00,0.00,0.00,0.00,0.00,52.00
city,27579.0,796.11,432.87,0.00,409.00,834.00,1096.00,1649.00
customer_nw_category,28382.0,2.23,0.66,1.00,2.00,2.00,3.00,3.00
branch_code,28382.0,925.98,937.80,1.00,176.00,572.00,1440.00,4782.00
current_balance,28382.0,7380.55,42598.71,-5503.96,1784.47,3281.26,6635.82,5905904.03
previous_month_end_balance,28382.0,7495.77,42529.35,-3149.57,1906.00,3379.92,6656.54,5740438.63
average_monthly_balance_prevQ,28382.0,7496.78,41726.22,1428.69,2180.95,3542.86,6666.89,5700289.57


## 8. Categorical Column Summary

Review the key categorical columns and their unique values before cleaning.


In [8]:
# Identify categorical columns in the raw dataset.
categorical_cols = df.select_dtypes(include='object').columns.tolist()

# Print unique value counts and sample values for categorical columns.
for col in categorical_cols:
    print(f'Column: {col}')
    print(f'Unique Values: {df[col].nunique(dropna=True)}')
    print(df[col].value_counts(dropna=False).head(10))
    print('-' * 50)



Column: gender
Unique Values: 2
gender
Male      16548
Female    11309
NaN         525
Name: count, dtype: int64
--------------------------------------------------
Column: occupation
Unique Values: 5
occupation
self_employed    17476
salaried          6704
student           2058
retired           2024
NaN                 80
company             40
Name: count, dtype: int64
--------------------------------------------------
Column: last_transaction
Unique Values: 361
last_transaction
NaT           3223
2019-12-31    1672
2019-12-28     831
2019-12-17     654
2019-12-27     632
2019-12-25     596
2019-12-26     576
2019-12-24     561
2019-12-20     486
2019-12-18     478
Name: count, dtype: int64
--------------------------------------------------


## 9. Extraction Summary

The raw dataset has been loaded and validated. Key observations from extraction:

1. The dataset contains **28,382 customers** and **21 columns**.
2. The target variable is `churn`, with an overall churn rate of about **18.53%**.
3. Missing values exist mainly in `dependents`, `city`, `gender`, and `occupation`.
4. No duplicate rows were found in the raw dataset.
5. Numeric balance and transaction fields show wide ranges, so outlier handling is needed in the cleaning stage.

The dataset is ready for the next notebook: **02_cleaning.ipynb**.
